<h1 style = "color : #0e68ddff; text-align : center;"><em>Where should I live?</em> - Building an Interactive Map Notebook</h1>
<p style = "font-size : 16px; text-align: center;">The goal of this section is to create an interactive map of Europe where users can explore cities and view relevant information</p>
<br>
<p style = "font-size : 12px; text-align: center;"><b>NOVA IMS</b></p>
<p style = "font-size : 10px; text-align: center;">Programming for Data Science</p>
<p style = "font-size : 10px; text-align: center;">Diogo Gonçalves, João Marques, Juan Mendes & Gustavo Franco</p>
<br>

<h2  style = "color : #0e68ddff;"> Imports</h2>

In [222]:
import pandas as pd
import plotly.graph_objects as go
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import re
import time
import warnings
warnings.filterwarnings('ignore')

In [208]:
#!pip install requests

In [209]:
#!pip install beautifulsoup4

In [210]:
#!pip install selenium

<hr style = "border: 3px solid #0e68ddff;">
<h2 style = "color : #0e68ddff;">Dataset Importing & Preparation</h2>
<p style = "font-size : 15px;">Reading of dataset from <code>city_data_clean.csv</code> file that we have prepared previously.</p>

In [211]:
city_data = pd.read_csv("city_data_clean.csv")
city_data.head()

,Population Density,Population,Working Age Population,Youth Dependency Ratio,Unemployment Rate,GDP per Capita,Days of very strong heat stress,Average Monthly Salary,Average Rent Price,Average Cost of Living,...,Scots,Serbian,Slovak,Slovene,Spanish,Swedish,Turkish,Unknown,Urdu,Valencian
0,310.0,2983513,2018818,20.1,10.2,55770.0,3,2500,1050,2061,...,0,1,0,0,0,0,1,0,0,0
1,243.0,375489,250472,20.3,3.0,66689.0,0,3200,1100,2186,...,0,0,0,0,0,0,0,0,0,0
2,681.0,3284548,2137425,27.5,10.7,62500.0,3,3350,1200,1900,...,0,0,0,0,0,0,0,0,0,0
3,928.0,1139663,723396,27.7,6.2,57595.0,3,2609,900,1953,...,0,0,0,0,0,0,0,0,0,0
4,552.0,645813,417832,24.8,5.3,53311.0,2,2400,827,1200,...,0,0,0,0,0,0,0,0,0,0


<p>As we are not using all the columns in the original cleaned dataset, we can copy it and filter only the columns we need for this notebook.</p>

In [212]:
imdf = city_data[['City', 'Country', 'Population', 'Average Monthly Salary', 'Average Cost of Living']].copy()
imdf.head()

,City,Country,Population,Average Monthly Salary,Average Cost of Living
0,Vienna,Austria,2983513,2500,2061
1,Salzburg,Austria,375489,3200,2186
2,Brussels,Belgium,3284548,3350,1900
3,Antwerp,Belgium,1139663,2609,1953
4,Gent,Belgium,645813,2400,1200


<hr style = "border: 3px solid #0e68ddff;">
<h2  style = "color : #0e68ddff;">Web Scraping</h2>
<p>For the web scrapping of the coordinates part, we have defined a strategy, implemented using functions, to retrieve the coordinates starting from <code>https://en.wikipedia.org/wiki/Main_Page</code> using <code>Selenium</code> (to navigate the wikipedia website and get to the pages we want) and <code>BeautifulSoup</code> (to help extract the coordinates from each page).


1. Initialization of the 2 new columns for <code>Latitude</code> and <code>Longitude</code>:

In [213]:
imdf["Latitude"] = None
imdf["Longitude"] = None

2. Definition of an inner function that is able to navigate the wikipedia, go to the right page of each city and retrieve the coordinates.


<p>While searching for the cities we followed a general case of searching: '[City] [Country]'. This approach has 3 major excecptions:</p>
<ul><li>Direct Hit Cases: when on Wikipedia, when we search popular terms, it will redirect us directly to a page. We don't want a specific page, we want to search, retrieve the links corresponding to the search and go through them until we find the coordinates, so for this cases we had to search <b>'[City] city [Country]'</b> in order to get a list of links and don't skip it and go directly to the page suggested by Wikipedia. This case is applied in popular cities like Rome, Paris, etc and Luxembourg (more than popular, it is in a country with the same name, 'Luxembourg' is the official name both for country and city, but Wikipedia has the name of the city page as 'Luxembourg City');</li>
<li>Ostrava Case: in this case, the festival 'Colours of Ostrava' is actually more popular on Wikipedia than the city, and as this festival has designated coordinates, we would be retrieving the coordinates of the festival. To avoid this, we search for <b>'[City] city'</b> to make sure we are extracting the right coordinates;</li>
<li>Turkiye Case: although the official name of this country is 'Turkiye', Wikipedia only recognises it as beeing 'Turkey'. When we use the offical name of the country and the cities name, wikipedia goes to general 'Turkey' page because only there are the names of the cities and 'Turkiye' referenced - the pages of the cities have no reference of 'Turkiye', so we have to search for <b>[City] Turkey</b>.</li></ul>

In [214]:
def get_coordinates(browser, city, country):
        """Inner function to navigate inside Wikipedia and extract the coordinates of each city:
        1. Find search icon on browser and click it
        2. Find Search Bar and Search for the city page on wikipedia
            General case: '[City] [Country]'
            Direct hit cases: '[City] city [Country]' for Rome, Paris, Berlin, Geneva, Luxembourg, Adana
            Ostrava case: '[City] city'
            Turkiye case: '[City] Turkey'
        3. Extract the latitude and longitude from the city's wikipedia page using BeautifulSoup
            3.1. If found, retrieve the rest of the links on search results and loop through them (max 5 tries) until find coordinates
        4. Return to main page to continue the outer function loop
        """
        
        
        #1.
        search_icon_input = browser.find_element(By.CLASS_NAME, "mw-ui-icon-search") #Find seach icon
        search_icon_input.click() #Click search icon
        time.sleep(1) #Wait for 1 second - just to be sure the site is all loaded
        
        #2. 
        search_bar_input = browser.find_element(By.CLASS_NAME, "cdx-text-input__input") #find search bar input

        # Direct Hit cases
        if city == 'Rome' or city == 'Paris' or city == 'Berlin' or city == 'Geneva' or city == 'Luxembourg' or city == 'Adana':
            search_bar_input.send_keys(f"{city} city {country}") #searching for the direct hits cases

        # Ostrava case
        elif city == 'Ostrava':
            search_bar_input.send_keys(f"{city} city") #searching for the Ostrava case

        # Turkiye case
        elif country == 'Turkiye':
            search_bar_input.send_keys(f"{city} Turkey") #searching for the Turkiye case

        # General case
        else:
            search_bar_input.send_keys(f"{city} {country}") #searching for the general case

        search_bar_input.send_keys(Keys.RETURN) #pressing enter to search
        time.sleep(2) #Waiting for 2 seconds to be sure the page is loaded
        
        #3. 
        html = browser.page_source #getting the html of the page
        readable_html = BeautifulSoup(html, "html.parser") #making it readable with BeautifulSoup
    
        latitude = readable_html.find("span", {"class": "latitude"}) #trying to find latitude
        longitude = readable_html.find("span", {"class": "longitude"}) #trying to find longitude
        
        links = browser.find_elements(By.CSS_SELECTOR, ".mw-search-result-heading a") #getting all the links from search results

        if latitude and longitude:
            return latitude.text, longitude.text
        
        #3.1.
        max_tries = 5 #setting max tries to 5
        
        for link in links[:max_tries]: #looping through the links (max 5 tries)
            link.click() #clicking the link
            time.sleep(2) #waiting for 2 seconds to be sure the page is loaded
    
            soup = BeautifulSoup(browser.page_source, "html.parser") #getting the html of the page with BeautifulSoup
            latitude = soup.find("span", class_="latitude") #finding latitude
            longitude = soup.find("span", class_="longitude") #finding longitude
    
            if latitude and longitude: #if both latitude and longitude are found
                return latitude.text, longitude.text #returning the values
            else:
                return None, None #returning None if not found

        #4.
        browser.back() #going back to the search results page
        time.sleep(2) #waiting for 2 seconds to be sure the page is loaded
        #READY TO CONTINUE THE OUTER LOOP

3. Designing the outer function that is reponsible for the 'main' Selenium functions - opening and closing browser - as well as looping through each city

In [215]:
def outer_function_coordinates(data):
    """Main function to navigate Wikipedia and run a loop to get to the city's pages and extract coordinates:
    1. Open Google Chrome 
    2. Search Wikipedia
    3. Loop to insert the coordenates in the dataset (using inner function get_coordinates)
        3.1. Get coordinates for each city
        3.2. Insert coordinates into the dataset
    4. Close browser
    """
    #1.
    browser = webdriver.Chrome() #opening Chrome

    #2.
    browser.get('https://en.wikipedia.org/wiki/Main_Page') #going to Wikipedia, as requested by guidelines

    time.sleep(1) #waiting for the page to load

    #3.
    for idx in data.index: #looping using idexes of the dset
        city = data.loc[idx, "City"] #retrieving city name
        country = data.loc[idx, "Country"] #retrieving country name

        #3.1.
        latitude, longitude = get_coordinates(browser ,city, country) #using inner function and storing results in auxiliary variables

        #3.2.
        data.loc[idx, "Latitude"] = latitude #insertinf Latitude into the dset
        data.loc[idx, "Longitude"] = longitude #inserting Longitude into the dset

    #4.
    browser.quit() #DONE

4. We just apply the outer function to the data set to fill <code>Latitude</code> and <code>Longitude</code>:

In [ ]:
outer_function_coordinates(imdf) # applying the main function to the dataset

5. Run some checks on this finalised dataset ready to build the Interactive Map:

In [ ]:
imdf.head() #check

,City,Country,Population,Average Monthly Salary,Average Cost of Living,Latitude,Longitude
0,Vienna,Austria,2983513,2500,2061,48°12′30″N,16°22′21″E
1,Salzburg,Austria,375489,3200,2186,47°48′00″N,13°02′42″E
2,Brussels,Belgium,3284548,3350,1900,50°50′48″N,04°21′09″E
3,Antwerp,Belgium,1139663,2609,1953,51°13′04″N,04°24′01″E
4,Gent,Belgium,645813,2400,1200,51°03′13″N,03°43′31″E


In [ ]:
imdf.info() #check for missing values in the newly created columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   City                    84 non-null     object
 1   Country                 84 non-null     object
 2   Population              84 non-null     int64 
 3   Average Monthly Salary  84 non-null     int64 
 4   Average Cost of Living  84 non-null     int64 
 5   Latitude                84 non-null     object
 6   Longitude               84 non-null     object
dtypes: int64(3), object(4)
memory usage: 4.7+ KB


<hr style = "border: 3px solid #0e68ddff;">
<h2 style = "color : #0e68ddff;">Interactive Map</h2>
<p style = "font-size : 16px;">This part is dedicated to the construction of the interactive map, using <code>Plotly Graph Objects</code> (this was a chosen approach instead of <code>plotly.express.scatter_geo</code> so that we could have more ways of customising our map, and instead of using <code>geopandas</code> as we plan to use it in the next section of the project).</p>
<br>

<p>Firstly we have to prepare the coordinates for the map construction. This is, most of coordinates are on Wikipedia like <code>XXºYY'ZZ''N XXºYY'ZZ''W</code> but some of them are not exactly like this. They could be more like <code>XXºYY'N XXºW</code>, for example, not having the full coordinates and having some kind of decimals in the minutes or seconds. For the Interactive Map we need to have everything in degrees with decimals, so we will use <code>regex</code> to identify this paterns and uniformise all the coordinates.</p>

In [260]:
def interactive_coordinates(str_coordinates):
    """Function that uniformises coordinates taken from Wikipedia:
    1. Uses regex to extract degrees, minutes, seconds and direction
    2. Converts Degrees-Minutes-Seconds (DMS) to Decimal Degrees
    3. Returns Decimal Degrees"""
    
    # Degrees: (\d+)° -> Integer only
    # Minutes: (\d+(?:\.\d+)?)′ 
    # Seconds: (?:(\d+(?:\.\d+)?)″)?
    pattern = r"(\d+)°\s*(\d+(?:\.\d+)?)′\s*(?:(\d+(?:\.\d+)?)″)?\s*([NSEW])" #creating the regex pattern
    
    match = re.search(pattern, str_coordinates) #searching for the pattern in the string

    deg, min, sec, direction = match.groups() #extracting the groups from the match


    dd = float(deg) + float(min)/60 + (float(sec)/3600 if sec else 0) #converting Degress-Minutes-Seconds to Decimal Degrees

    if direction in ('S', 'W'):
        dd = -dd
        
    return dd

In [ ]:
imdf["Interactive Latitude"] = imdf["Latitude"].apply(interactive_coordinates) #converting Latitude to Decimal Degrees
imdf["Interactive Longitude"] = imdf["Longitude"].apply(interactive_coordinates) #converting Longitude to Decimal Degrees

In [ ]:
imdf.head() #check

,City,Country,Population,Average Monthly Salary,Average Cost of Living,Latitude,Longitude,Interactive Latitude,Interactive Longitude
0,Vienna,Austria,2983513,2500,2061,48°12′30″N,16°22′21″E,48.208333,16.372500
1,Salzburg,Austria,375489,3200,2186,47°48′00″N,13°02′42″E,47.800000,13.045000
2,Brussels,Belgium,3284548,3350,1900,50°50′48″N,04°21′09″E,50.846667,4.352500
3,Antwerp,Belgium,1139663,2609,1953,51°13′04″N,04°24′01″E,51.217778,4.400278
4,Gent,Belgium,645813,2400,1200,51°03′13″N,03°43′31″E,51.053611,3.725278


<p>Right after this, we are ready to build the map!</p>

In [270]:
fig = go.Figure(data = go.Scattergeo( #inside a Figure of Plotly Graph Objects, make a Scattergeo plot
                lon=imdf["Interactive Longitude"], #draw longitude using the new prepared longitude column
                lat=imdf["Interactive Latitude"], #draw latitude using the new prepared latitude column
                text=imdf["City"], #draw city names
                mode="markers+text", #get to plot points and city names
                textposition="bottom left", #put the names in the bottom left of the markers
                marker=dict(size=8, color="#0e90e7", line=dict(color='white', width=1), opacity=0.7), #customizing markers
                customdata = imdf[["Country", "Population", "Average Monthly Salary", "Average Cost of Living"]].values, #specifiyng the data we need to put on our map while hovering
                hovertemplate=( #desigining the hover template with HTML tas
                    "<b>%{text}</b><br>" +
                    "Country: %{customdata[0]}<br>" +
                    "Population: %{customdata[1]}<br>" +
                    "Average Monthly Salary: %{customdata[2]}<br>" +
                    "Average Cost of Living: %{customdata[3]}<extra></extra>")))



fig.update_layout(
        title=dict(
            text="<b><i>Where Should I Live?</i> - Map of European Cities</b><br><i>Hover over the markers for more info</i>", #editing title with HTML tags
            font=dict(size=24, color="#0e68dd") #editing title 
        ),
            geo=dict(
            scope="world", #have the whole world - 'europe' would not show enough area for our cities
            projection_type="orthographic", #making our map a globe! LOOKS BEUTIFULL
            showcountries=True, #show borders
            countrycolor="#000000", #black borders
            landcolor="white",
            showocean=True, #get an ocean
            oceancolor="#0e90e7", #make the ocean blue
            lataxis_range=[30, 72], # defining latitude range to aut focus on europe
            lonaxis_range=[-15, 45], #the same for longitude
        ),
    height=1000 #specifying the height of the figure
)

fig.show()